In [1]:
import numpy as np 
from matplotlib import pyplot as plt 
from scipy.interpolate import interp1d
%load_ext autoreload
%autoreload 2
%matplotlib inline


## Functions that generates the dirichlet distribution

In [2]:
def draw_values(data, num=1, ret_mids=False): 
    if num==1: 
        samp = np.random.dirichlet(data[:, 1])
        samp = samp/np.trapz(samp, data[:, 0])
       
    else:
        samp = np.random.dirichlet(data[:, 1], size=num)
        samp = np.array([el/np.trapz(el, data[:, 0]) for el in samp])
    
    if ret_mids == False: 
        return samp
    else:
        return data[:, 0], samp

In [3]:
def inter_extrapolate(zbins, sample_zbin, sample_nz):
    interp_function  = interp1d(sample_zbin, sample_nz, fill_value = 0.0)

In [4]:
nz_alpha_folder = './alphas/'
data_0_3_0_6 = np.loadtxt(nz_alpha_folder+'result_photometry_0_3_0_6_alpha.dat')
data_0_6_0_9 = np.loadtxt(nz_alpha_folder+'result_photometry_0_6_0_9_alpha.dat')
data_0_9_1_2 = np.loadtxt(nz_alpha_folder+'alpha_result_0_9_1_2.dat')
data_1_2_1_5 = np.loadtxt(nz_alpha_folder+'alpha_result_1_2_1_5.dat')

## You can specify any z-binning you want, this is an example

In [5]:
zbound = np.linspace(0.0,6.001,300)
zbin = (zbound[1:]+zbound[:-1])/2
bin_width = zbin[1]-zbin[0]

Generate 10000 samples

In [6]:
n_realization = 10000

sample_0_3_0_6 = draw_values(data_0_3_0_6, n_realization)
sample_0_6_0_9 = draw_values(data_0_6_0_9, n_realization)
sample_0_9_1_2 = draw_values(data_0_9_1_2, n_realization)
sample_1_2_1_5 = draw_values(data_1_2_1_5, n_realization)

Interpolate the generated distribution

In [7]:
interp_function1  = interp1d(data_0_3_0_6[:,0], sample_0_3_0_6, fill_value = 0.0, kind = 'linear', bounds_error = False)
interp_function2  = interp1d(data_0_6_0_9[:,0], sample_0_6_0_9, fill_value = 0.0, kind = 'linear', bounds_error = False)
interp_function3  = interp1d(data_0_9_1_2[:,0], sample_0_9_1_2, fill_value = 0.0, kind = 'linear', bounds_error = False)
interp_function4  = interp1d(data_1_2_1_5[:,0], sample_1_2_1_5, fill_value = 0.0, kind = 'linear', bounds_error = False)


Put them into one numpy array

In [8]:


nz_realization_array = np.array([interp_function1(zbin)[:]/np.sum(interp_function1(zbin))/bin_width,
                       interp_function2(zbin)[:]/np.sum(interp_function2(zbin))/bin_width,
                       interp_function3(zbin)[:]/np.sum(interp_function3(zbin))/bin_width,
                       interp_function4(zbin)[:]/np.sum(interp_function4(zbin))/bin_width ]).T
nz_realization_array = np.swapaxes(nz_realization_array,0,1)
nz_realization_array = np.swapaxes(nz_realization_array,1,2)

In [9]:
# The shape is (# of realization, # of tomographic bins, # of z bins)
nz_realization_array.shape

(10000, 4, 299)

In [10]:
nz_average = np.mean(nz_realization_array,axis = 0)

## Show the average distribution and the first 200 realizations

In [ ]:
plt.plot(zbin,nz_average[0,:], color = 'blue',alpha = 0.5 )
plt.plot(zbin,nz_average[1,:], color = 'red',alpha = 0.5 )
plt.plot(zbin,nz_average[2,:], color = 'green',alpha = 0.5)
plt.plot(zbin,nz_average[3,:], color = 'yellow',alpha = 0.5)

#show 
for i in range(200):
    nz_realization = nz_realization_array[i,:,:]
    plt.plot(zbin,nz_realization[0,:], color = 'blue', alpha = 0.05)
    plt.plot(zbin,nz_realization[1,:], color = 'red', alpha = 0.05)
    plt.plot(zbin,nz_realization[2,:], color = 'green', alpha = 0.05)
    plt.plot(zbin,nz_realization[3,:], color = 'yellow', alpha = 0.05)

plt.xlim(0,2)

## You could save nz_realization_array however you want